# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an interactive template for loading and exploring the FAIR² dataset using the `mlcroissant` library. We will use Croissant metadata and reference all entities (record sets, fields, columns) by their canonical `@id` values as defined in the schema.

### Dataset Source
Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata with `mlcroissant`, inspect the title and description, and check dataset structure.

In [ ]:
import mlcroissant as mlc
import pandas as pd

croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")
print(f"License: {dataset.metadata.license}")

"""
All navigation through record sets, fields, and columns will use their `@id`. 
Let's inspect the available record sets in this package.
"""

## 2. Data Overview
Explore available record sets, their `@id`s, and the fields/columns within each set.

In [ ]:
# The dataset might have multiple record sets; let's enumerate them with their @id.
print("Available Record Sets and Fields (by @id):\n")
record_sets = dataset.metadata.recordSets
if not record_sets:
    print("No record sets declared in the top-level schema. Attempting to infer from files and fields...")
    # Fallback in case Croissant schema doesn't have explicit recordSets property but only has fileObjects
    if hasattr(dataset.metadata, 'fileObjects'):
        for fileobj in dataset.metadata.fileObjects:
            rid = fileobj['@id'] if isinstance(fileobj, dict) else getattr(fileobj, '@id', None)
            print(f"- FileObject @id: {rid}")
    else:
        print("No fileObjects found either.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        if 'fields' in rs:
            for f in rs['fields']:
                fid = f['@id'] if isinstance(f, dict) else f
                print(f"  - Field: {fid}")
        if 'columns' in rs:
            for c in rs['columns']:
                cid = c['@id'] if isinstance(c, dict) else c
                print(f"  - Column: {cid}")
# If recordSet is empty list, let's inspect as Croissant 1.x will have schema in fileObjects
if not getattr(dataset.metadata, 'recordSets', None) and getattr(dataset.metadata, 'fileObjects', None):
    from pprint import pprint
    print("\nFile Objects (often used as RecordSets):")
    for fo in dataset.metadata.fileObjects:
        print(f"FileObject @id: {fo['@id']}\n  Name: {fo.get('name', '')}")
        if 'fields' in fo:
            print("  Fields:")
            # These fields are likely to be @ids
            for field in fo['fields']:
                pprint(field)
        print()

## 3. Data Extraction
Identify the record set `@id` of interest and load its data into a DataFrame for analysis. We will reference every entity by `@id` as required.

For the current schema, if explicit `recordSet` or `fileObject` entities are not present, consult the documentation or Croissant file to find the downloadable table structure. For illustration, we will list all record set files and load data from the first available set.

In [ ]:
# Find all available record set/fileobject IDs for extraction
from pprint import pprint

record_set_ids = []
if getattr(dataset.metadata, 'recordSets', None):
    for rs in dataset.metadata.recordSets:
        record_set_ids.append(rs['@id'])
elif getattr(dataset.metadata, 'fileObjects', None):
    # Croissant v1.0: fileObjects are typically record sets
    for fo in dataset.metadata.fileObjects:
        record_set_ids.append(fo['@id'])
else:
    print('No accessible record sets or fileObjects found!')

print("Record sets (for extraction):")
pprint(record_set_ids)

# Select primary record set for demo (edit as needed)
record_set_id = record_set_ids[0] if record_set_ids else None
assert record_set_id, "No record set available to extract!"

dataframes = {}
for rsid in record_set_ids:
    print(f"Loading records for {rsid} ...")
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)
    print(f"==> DataFrame columns for {rsid}:")
    print(dataframes[rsid].columns.tolist())
    print(f"Number of records: {len(dataframes[rsid])}\n")
# Inspect primary set
print(f"Showing first 5 rows of record set {record_set_id}:")
dataframes[record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let’s select a numeric field (by column `@id`/name), filter and normalize data, and demonstrate grouping.

In [ ]:
df = dataframes[record_set_id]
print("DataFrame Shape:", df.shape)

# Choose column names - for demonstration, select first numeric column with more than a few non-null entries
numeric_field = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]) and df[col].notnull().sum() > 0:
        numeric_field = col
        break
if numeric_field is None:
    numeric_field = df.columns[0] # fallback

print(f"Chosen numeric field for analysis: {numeric_field}\n")
mean_val = df[numeric_field].mean()
threshold = mean_val if pd.notnull(mean_val) else 0

filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with `{numeric_field}` > {threshold}:")
print(filtered_df.head())

# Normalization (z-score)
filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"\nNormalized `{numeric_field}` for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# If there is a likely categorical field, group by it
candidate_cats = [c for c in df.columns if pd.api.types.is_object_dtype(df[c])]
group_field = candidate_cats[0] if candidate_cats else None
if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Let’s plot the distribution of the numeric field and (if available) the per-group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

if group_field is not None:
    # Show group means if group_field exists
    plt.figure(figsize=(10,4))
    order = filtered_df[group_field].value_counts().index
    sns.barplot(y=group_field, x=numeric_field, data=filtered_df, ci=None, order=order)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
We have demonstrated how to:
- Load and parse the dataset using `mlcroissant` based on its Croissant schema URL.
- Reference all entities (record sets, fields/columns) by their `@id` per best FAIR practice.
- Explore schema, load tabular data, and conduct basic filtering, normalization, grouping, and visualization.

Further analysis can be carried out using the DataFrames or the rich Croissant metadata objects.